# geo0162 – Data models in code

**Unit:** geo0160 Data models & formats · **Slides:** `geo0160_Data_Models_Formats.pdf` (session folder) · **Data:** your station GeoPackage from session 1, 16 tiles (4 km × 4 km) of the NRW terrain model DGM1 around the Fürstenberg and the Hees near Xanten (downloaded below, ≈ 34 MB)

In QGIS a layer is either vector or raster. In code you see what that means: a **vector** layer is a table with a geometry column and a CRS; a **raster** is a grid of numbers *plus* a transform that puts the grid on the ground.

You will learn to
1. read the **geometry type, CRS, extent and attribute types** of a vector layer,
2. write and read geometries as **WKT**,
3. read the **properties of a raster**: CRS, transform, resolution, extent, data type, nodata,
4. **merge** raster tiles into one mosaic,
5. convert between **map coordinates (E, N)** and **cells (row, col)**,
6. derive a **hillshade** from the terrain model, and tell data from visualisation,
7. check the layers in **QGIS**.

## 0. Setup

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import shutil
import subprocess

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import shapely
from rasterio.merge import merge
from rasterio.plot import show
from rasterio.transform import rowcol, xy

# vector: your station GeoPackage from session 1 (geo0641)
GPKG = Path("../s01_2026-10-05/data/derived/dwd_stations.gpkg")
LAYER = "kl_stations_nrw_active"

# raster: 4 x 4 tiles of 1 km x 1 km of the NRW terrain model DGM1 (1 m): the Hees and the Fürstenberg near Xanten
# name = dgm1_32_<easting of the lower left corner in km>_<northing in km>_1_nw_<year>.tif
SCENE = (322000, 5722000, 326000, 5726000)       # left, bottom, right, top (EPSG:25832), corners on a 2 km grid
TILES = [f"dgm1_32_{e}_{n}_1_nw_2025.tif"
         for e in range(SCENE[0] // 1000, SCENE[2] // 1000)
         for n in range(SCENE[1] // 1000, SCENE[3] // 1000)]
URL = "https://www.opengeodata.nrw.de/produkte/geobasis/hm/dgm1_tiff/dgm1_tiff/"
SHARED = Path("/shared/data/opengeo/geo0160")       # read-only copy on the JupyterHub, if present
DOWNLOAD = Path("../data/dgm1")                   # top-level data/ of your clone (ignored by git)
OUT = Path("data/derived")                        # results of this notebook (ignored by git)
DOWNLOAD.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)


def get_tile(name):
    """Path of one DGM1 tile: shared copy if present, otherwise download once."""
    if (SHARED / name).exists():
        return SHARED / name
    path = DOWNLOAD / name
    if not path.exists():
        print("downloading", name)
        urlretrieve(URL + name, path)
    return path


print("GeoPackage found:", GPKG.exists())
print(len(TILES), "tiles:", TILES[0], "...", TILES[-1])

## 1. Vector: your DWD stations
### 1.1 What is in the layer?
**Predict:** which geometry type and which CRS do you expect? Roughly which extent (in which unit)?

In [ ]:
if not GPKG.exists():
    raise FileNotFoundError(f"{GPKG} not found: run geo0641 (session 1) first, or adapt GPKG")
print(gpd.list_layers(GPKG))
stations = gpd.read_file(GPKG, layer=LAYER)
print(len(stations), "features")
print("geometry types:", stations.geom_type.unique())
print("CRS:", stations.crs, "-", stations.crs.name)
print("extent (xmin, ymin, xmax, ymax):", stations.total_bounds.round(0))

### 1.2 Attributes and their types
**Predict:** which type does each field have: text, integer, decimal, date?

In [ ]:
stations.dtypes

In [ ]:
ids = stations["station_id"]
if not ids.map(lambda v: isinstance(v, str)).all():
    print("station_id is not text: run s01_2026-10-05/geo0641_DWD_Station_Pipeline_fix1.ipynb (see errata.md)")
else:
    print("station_id is text, e.g.", ids.iloc[0], "- identifiers keep their leading zeros")

### 1.3 Geometry as text: WKT
*Well-known text* (WKT) is the OGC text form of a geometry; PostGIS, QGIS and Shapely all read and write it.

**Predict:** what does `POINT (x y)` hold here: longitude/latitude or easting/northing?

In [ ]:
first = stations.iloc[0]
print(first["station_id"], first.geometry.wkt)
print("in EPSG:4326:", stations.to_crs("EPSG:4326").geometry.iloc[0].wkt)

A polygon from WKT: a rough box around Castra Vetera on the Fürstenberg (EPSG:25832, metres). We use it again in §5 and §6.

In [ ]:
box = shapely.from_wkt(
    "POLYGON ((324500 5723800, 325200 5723800, 325200 5724800, 324500 5724800, 324500 5723800))")
print(box.geom_type, "-", round(box.area / 10_000, 1), "ha")

**Exercise:** which active DWD station lies closest to the box, and how far away is it (km)? Why can you compute the distance directly here, and when could you not?

In [ ]:
# YOUR CODE HERE


## 2. Raster: one tile of the terrain model
### 2.1 The transform
A raster stores only the cell values. The **transform** (six numbers) says where the grid lies: cell size in x and y, rotation terms, and the coordinates of the upper left corner.

**Predict:** read the transform of the first tile. What are the cell size and the upper left corner? How large is the tile if it has 1000 × 1000 cells? Compare with the tile name (which corner does the name give?).

In [ ]:
paths = [get_tile(t) for t in TILES]
src = rasterio.open(paths[0])
print(paths[0].name)
src.transform

In [ ]:
print("CRS:       ", src.crs)
print("resolution:", src.res)
print("extent:    ", src.bounds)
print("size:      ", src.width, "x", src.height, "cells,", src.count, "band(s)")
print("data type: ", src.dtypes[0], "  nodata:", src.nodata)

**Exercise:** compute the extent (left, bottom, right, top) yourself from `src.transform`, `src.width` and `src.height`, without `src.bounds`. Why is the y cell size negative?

In [ ]:
a = src.transform
# YOUR CODE HERE
print((left, bottom, right, top), "==", tuple(src.bounds))
src.close()

## 3. All 16 tiles: same grid, different place
Only the corner in the transform changes from tile to tile; cell size, CRS, data type and nodata are the same. That is what makes merging simple.

In [ ]:
for p in paths[:5]:                        # the first five are enough to see the pattern
    with rasterio.open(p) as s:
        print(p.name, " upper left:", (s.transform.c, s.transform.f), " res:", s.res, " ", s.crs, s.dtypes[0], s.nodata)

## 4. Merge the tiles into one mosaic
`rasterio.merge.merge` puts the tiles into one array and computes a new transform.

**Predict:** shape of the mosaic? Its upper left corner? Which number of the transform changes, which stay?

In [ ]:
srcs = [rasterio.open(p) for p in paths]
mosaic, transform = merge(srcs)            # mosaic: (bands, rows, cols)
profile = srcs[0].profile
for s in srcs:
    s.close()

print("shape:", mosaic.shape, "  memory:", round(mosaic.nbytes / 1e6), "MB")
print(transform)
dem = np.ma.masked_equal(mosaic[0], profile["nodata"])     # hide nodata cells
print("elevation:", round(float(dem.min()), 1), "-", round(float(dem.max()), 1), "m")

Save the mosaic as a GeoTIFF: same profile as the tiles, new size and transform.

In [ ]:
MOSAIC = OUT / "dgm1_fuerstenberg_hees_mosaic.tif"
profile.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=transform,
               compress="deflate", tiled=True, blockxsize=256, blockysize=256)
with rasterio.open(MOSAIC, "w", **profile) as dst:
    dst.write(mosaic)
print("written:", MOSAIC, round(MOSAIC.stat().st_size / 1e6, 1), "MB")

*Alternatives:* in QGIS *Raster ▸ Miscellaneous ▸ Merge…* or a virtual raster (*Build Virtual Raster…*, a small `.vrt` file that only points to the tiles). On the command line: `gdal_merge.py -o mosaic.tif tile1.tif tile2.tif …` or `gdalbuildvrt mosaic.vrt *.tif`.

## 5. From map coordinates to cells and back
### 5.1 Which cell holds the fortress?
The centre of the legionary fortress *Vetera I* (geo0241) lies at about E 324 843, N 5 724 312 (EPSG:25832).

**Predict:** in which row and column of the mosaic is it? (Upper left corner and cell size are enough.)

In [ ]:
E, N = 324843, 5724312
row, col = rowcol(transform, E, N)         # x (easting) first!   (open file: src.index(E, N))
print("row, col:", row, col)
print("elevation:", round(float(dem[row, col]), 2), "m")
print("cell centre back in E, N:", xy(transform, row, col))

### 5.2 Easting and northing swapped
**Predict:** what happens if you give northing first?

In [ ]:
r, c = rowcol(transform, N, E)             # wrong order
print("row, col:", r, c)
try:
    print(dem[r, c])
except IndexError as err:
    print("IndexError:", err)

Row and column are far outside the 4000 × 4000 grid. **Map coordinates: x = easting first. Arrays: row first** (`dem[row, col]`).

## 6. Hillshade: from data to a picture
### 6.1 Elevation is data, a hillshade is a picture
The terrain model holds **elevations in metres**: you can compute with them (heights, slopes, volumes, flood depths). In a grey or colour ramp, however, a 1 m high bank on a 50 m high ridge is invisible.

A **hillshade** (*Schummerung*) is derived from the elevations: for every cell it computes how bright the surface would be under a lamp from a chosen direction (**azimuth**, clockwise from north) and height above the horizon (**altitude**). The result is a **brightness 0–255**: no metres, no unit. It depends on the lamp, not only on the terrain. Use it to *see* landforms, never to measure.

The formula needs the **slope** (how steep) and the **aspect** (which direction the cell faces) from the gradient of the DEM, divided by the cell size; `z` exaggerates the heights.

In [ ]:
def hillshade(dem, res, azimuth=315, altitude=45, z=1.0):
    """Brightness 0-255 of each cell under a lamp at azimuth/altitude (degrees)."""
    dem = np.asarray(dem, dtype="float32")  # float32 halves the memory of a 4000 x 4000 grid
    dy, dx = np.gradient(dem * z, res)     # change per row (towards south) and per column (towards east)
    aspect = np.arctan2(-dx, dy)           # direction the cell faces
    slope = np.arctan(np.hypot(dx, dy))    # steepness (radians)
    del dx, dy                             # free memory early
    az, alt = np.radians(azimuth), np.radians(altitude)
    shade = np.sin(alt) * np.cos(slope) + np.cos(alt) * np.sin(slope) * np.cos(az - aspect)
    return np.clip(255 * shade, 0, 255, out=shade)


res = transform.a                          # 1 m
z_filled = dem.astype("float32").filled(np.nan)   # nodata -> NaN, so it does not create artificial cliffs
hs = hillshade(z_filled, res)              # standard: light from the north-west, 45° above the horizon
print("hillshade range:", np.nanmin(hs).round(), "-", np.nanmax(hs).round(), "  a flat cell:", round(255 * np.sin(np.radians(45))))

**Predict:** the fortress centre had an elevation in metres (§5.1). What does the hillshade hold in the same cell?

In [ ]:
print("DEM:      ", round(float(dem[row, col]), 2), "m")
print("hillshade:", round(float(hs[row, col])), "(brightness, no unit)")

### 6.2 Compare: elevation, hillshade, and both together

In [ ]:
ext = (transform.c, transform.c + transform.a * dem.shape[1],
       transform.f + transform.e * dem.shape[0], transform.f)          # left, right, bottom, top
v = np.s_[::2, ::2]        # plot every 2nd cell: the screen has fewer pixels than 4000 x 4000, and it saves memory
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
axs[0].imshow(dem[v], extent=ext, cmap="terrain")
axs[0].set_title("DGM1: elevation (m)")
axs[1].imshow(hs[v], extent=ext, cmap="gray")
axs[1].set_title("hillshade, azimuth 315°, altitude 45°")
axs[2].imshow(hs[v], extent=ext, cmap="gray")
axs[2].imshow(dem[v], extent=ext, cmap="terrain", alpha=0.45)
axs[2].set_title("both: colour ramp over hillshade")
for ax in axs:
    gpd.GeoSeries([box], crs=profile["crs"]).boundary.plot(ax=ax, color="k", lw=0.8)
    ax.plot(E, N, "r+", ms=12)
    ax.ticklabel_format(useOffset=False, style="plain")
    ax.locator_params(nbins=5)
fig.suptitle("DGM1 Hees / Fürstenberg, 4 km × 4 km (Geobasis NRW, dl-de/zero-2-0)")
plt.show()

### 6.3 The lamp matters
**Predict:** what happens with the light from the **south-east** (azimuth 135°), and with a low sun (altitude 15°)?

In [ ]:
z2 = z_filled[v]                           # every 2nd cell = a 2 m grid: enough to compare the lamps, 4x less memory
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, (az, alt) in zip(axs, [(315, 45), (135, 45), (315, 15)]):
    ax.imshow(hillshade(z2, 2 * res, az, alt), extent=ext, cmap="gray")
    ax.set_title(f"azimuth {az}°, altitude {alt}°")
    ax.ticklabel_format(useOffset=False, style="plain")
    ax.locator_params(nbins=5)
plt.show()

With light from the south-east many people see valleys as ridges (*relief inversion*): our eyes expect light from the upper left. A low sun brings out low banks and ditches, but deepens the shadows. Cartographic standard: **315° / 45°**.

**Exercise:** compute the **slope in degrees** from the same gradient (`np.gradient`, cell size `res`). Where is the terrain steeper than 15°? What could the steep lines around the box be?

In [ ]:
# YOUR CODE HERE


### 6.4 Save the results
**GeoTIFF** keeps the numbers *and* the transform and CRS: GIS software places it on the map and you can compute with it. **PNG** is only a picture of a rendering (colours, no metres, no coordinates): use it in reports and slides.

In [ ]:
HILLSHADE = OUT / "dgm1_fuerstenberg_hees_hillshade.tif"
hs_profile = profile.copy()
hs_profile.update(dtype="uint8", nodata=0, count=1)
with rasterio.open(HILLSHADE, "w", **hs_profile) as dst:
    dst.write(np.nan_to_num(hs, nan=0).astype("uint8"), 1)
print("written:", HILLSHADE, round(HILLSHADE.stat().st_size / 1e6, 1), "MB")

Now the two renderings as PNG maps: one map each, with axes in km, legend or title, scale bar and source.

In [ ]:
def save_map(img, path, title, cmap, vmin=None, vmax=None, label=None):
    """Save one rendered raster as a PNG map (EPSG:25832, axes in km)."""
    fig, ax = plt.subplots(figsize=(10, 9), constrained_layout=True)
    im = ax.imshow(img[v], extent=ext, cmap=cmap, vmin=vmin, vmax=vmax)
    if label:
        fig.colorbar(im, ax=ax, shrink=0.8, label=label)
    gpd.GeoSeries([box], crs=profile["crs"]).boundary.plot(ax=ax, color="red", lw=2)
    x0, y0 = ext[1] - 650, ext[2] + 100                                   # scale bar 500 m
    ax.plot([x0, x0 + 500], [y0, y0], color="black", lw=5, solid_capstyle="butt")
    ax.text(x0 + 250, y0 + 40, "500 m", ha="center", va="bottom", fontsize=13,
            bbox=dict(fc="white", alpha=0.7, lw=0))
    km = plt.FuncFormatter(lambda v, _: f"{v / 1000:.1f}")
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(plt.MultipleLocator(1000))                 # a tick every km
        axis.set_major_formatter(km)
    ax.set_xlabel("E (km, EPSG:25832)")
    ax.set_ylabel("N (km, EPSG:25832)")
    ax.set_title(title, fontsize=15)
    fig.text(0.5, 0, "DGM1 © Geobasis NRW, dl-de/zero-2-0 · red: box from §1.3", ha="center", va="top", fontsize=11)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("written:", path, round(path.stat().st_size / 1e6, 1), "MB")


DTM_PNG = OUT / "dgm1_fuerstenberg_hees_dtm.png"
HILLSHADE_PNG = OUT / "dgm1_fuerstenberg_hees_hillshade.png"
save_map(dem, DTM_PNG, "DTM (DGM1, 1 m): elevation", "terrain", label="elevation (m above NHN)")
save_map(hs, HILLSHADE_PNG, "Hillshade (azimuth 315°, altitude 45°)", "gray", vmin=0, vmax=255)

The same in one command with GDAL (if installed: `gdaldem` on the command line, or *Raster ▸ Analysis ▸ Hillshade…* in QGIS, which calls it):

```
gdaldem hillshade -az 315 -alt 45 -z 1 -compute_edges dgm1_fuerstenberg_hees_mosaic.tif hillshade_gdal.tif
```

`gdaldem` uses a 3 × 3 window (Horn's formula) instead of `np.gradient`, so single cells differ slightly.

In [ ]:
if shutil.which("gdaldem"):
    out = OUT / "dgm1_fuerstenberg_hees_hillshade_gdal.tif"
    subprocess.run(["gdaldem", "hillshade", "-az", "315", "-alt", "45", "-z", "1", "-compute_edges",
                    str(MOSAIC), str(out)], check=True)
    with rasterio.open(out) as g:
        diff = np.abs(g.read(1).astype(float) - np.nan_to_num(hs))
    print("mean difference to our hillshade:", round(float(diff.mean()), 1), "of 255")
else:
    print("gdaldem not installed here - use QGIS: Raster > Analysis > Hillshade")

## 7. Check in QGIS
Open in QGIS (project CRS EPSG:25832): your `dwd_stations.gpkg`, the mosaic and the hillshade GeoTIFFs from `data/derived/` (on the hub: right-click → *Download*). Drag the PNG maps into QGIS too: what happens, and why?
1. Use **Identify** on a station, on a mosaic cell and on a hillshade cell: what does QGIS return in each case?
2. Style the mosaic with *Symbology ▸ Hillshade*: QGIS computes the hillshade on the fly for the screen, without a new file. Put the mosaic with a colour ramp and 50 % opacity over it.
3. Compare with the *DGM-Schummerung* WMS of Geobasis NRW (geo0921): a picture from a server. *Identify* returns colours, not heights.

In [ ]:
for p in (GPKG, MOSAIC, HILLSHADE, DTM_PNG, HILLSHADE_PNG):
    print(p.resolve())

## 8. Bridge to georeferencing
These six numbers put the grid on the ground. A scanned or photographed map has no transform yet: in the Georeferencer you set control points, and the transformation *Helmert* or *Polynomial 1* (affine) computes exactly these numbers.

In [ ]:
print("mosaic:", tuple(transform)[:6])

## 9. Wrap-up
1. **Vector:** geometry + attributes + CRS. Know the geometry type, the attribute types (identifiers stay text) and the CRS.
2. **WKT** is the common text form of geometries.
3. **Raster:** grid + transform + CRS. Resolution, extent, data type and nodata come with it. Tiles on the same grid merge into one mosaic with a new transform.
4. **x = easting first** for map coordinates, **row first** for arrays.
5. **DEM = data** (metres, measure with it); **hillshade = picture** derived from it (brightness, depends on the lamp, look at it).
6. **GeoTIFF** stores values + transform + CRS; **PNG** stores only the colours of a rendering.

**Exit ticket:** one sentence: which of the six transform numbers would change if the tiles had 2 m cells, and which if you added one more row of tiles in the north?